Offline teaching language view. Code, existing outputs and configuration retain the accepted reader identity. Fixed filing facts and reference figures are historical teaching inputs; new constructed calculations are not market-strategy reproduction.

[Entry modes and preparation](../NOTEBOOK_GUIDE.md). This stage delivers language views; portable dependencies and execution are checked in the later isolated cold start.

In [ ]:
from pathlib import Path
import os, sys
from uuid import uuid4
ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir())
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.chdir(ROOT)
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch08_regression_" + uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Default offline teaching; current empirical French execution SKIPPED (permissions unresolved).")
print("Run directory:", RUN_DIR)


# Chapter 8: regression, robust inference, and regularization

All observations below are **constructed teaching data**, not actual market observations or a CAPM empirical test. The synthetic daily process has true intercept 0.0001 return per day, slope 1.2, a market-dependent residual scale, and AR(1) residual dependence. It contains 400 equally spaced UTC daily timestamps. The first 280 are the training window; the last 120 are reserved for one final prediction-error evaluation.

## Goal and method

Fit a one-factor OLS model with an intercept, compare classical, HC3, and HAC standard errors, inspect a near-duplicate feature design, and compare Ridge, Lasso, and ElasticNet forecasts. Hyperparameters are selected only inside chronological `TimeSeriesSplit` folds with `gap=5`; each model is a `StandardScaler` + estimator `Pipeline`. The holdout is not used for selection. No trading performance or causal conclusion is drawn.

The OLS wrapper validates timezone-aware, unique, increasing timestamps; exact feature/target alignment; finite numeric inputs; nonconstant predictors; and full-rank design. Source notes for the official statsmodels and scikit-learn APIs are in `code/ch08_source_notes.md`.


## Observed run (seed 20261008)

On the 280-row training window, the constructed sample produced OLS α = 0.00064027 return/day, β = 1.124669, R² = 0.776925, and adjusted R² = 0.776123. The slope standard errors were 0.036144 (classical), 0.052391 (HC3), and 0.055402 (HAC, maxlags 5). The near-duplicate design condition number was 135,724. Training-only TimeSeriesSplit selected Ridge α = 0.01, Lasso α = 0.00001, and ElasticNet α = 0.00001 with `l1_ratio = 0.2`. The final 120-row constructed holdout MSEs were 0.0002291795 for the mean baseline, 0.0000400386 for one-factor OLS, 0.0000400437 for Ridge, 0.0000400481 for Lasso, and 0.0000400414 for ElasticNet, in squared daily-return units. These fixed-seed simulation values do not establish empirical forecast skill.


## 1. Setup and deterministic sample

In [ ]:
from pathlib import Path
import sys
ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "src" / "quant_course").is_dir())
sys.path.insert(0, str(ROOT / "src"))
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.dates import AutoDateLocator, ConciseDateFormatter
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error
from IPython.display import Image, display
from quant_course.regression_tools import (
    fit_ols_with_robust_covariance,
    select_regularized_models,
    simulate_capm_like_data,
)

SEED = 20261008
TRAIN_N = 280
HAC_MAXLAGS = 5
CV_GAP = 5
Y_UNIT = "Excess return (decimal per day)"
data = simulate_capm_like_data(n=400, seed=SEED, alpha=0.0001, beta=1.2, ar_phi=0.55)
training = data.iloc[:TRAIN_N]
holdout = data.iloc[TRAIN_N:]
print("CONSTRUCTED data only; no CAPM empirical test")
print(f"seed={SEED}; n={len(data)}; frequency=one synthetic daily observation")
print(f"train n={len(training)}: {training.index[0]} to {training.index[-1]}")
print(f"holdout n={len(holdout)}: {holdout.index[0]} to {holdout.index[-1]}")
print("true DGP alpha=0.0001 return/day; beta=1.2; residual has conditional heteroskedasticity and AR(1) phi=0.55")


## 2. OLS coefficient and covariance comparisons

The point estimate is the same for all covariance choices. Classical standard errors rely on the classical error assumptions; HC3 addresses heteroskedasticity, while HAC can address heteroskedasticity and serial dependence subject to additional conditions. The HAC lag count is fixed at 5 for this demonstration.

In [ ]:
training_features = training[["market_excess"]]
training_target = training["asset_excess"]
ols = fit_ols_with_robust_covariance(training_features, training_target, hac_maxlags=HAC_MAXLAGS)
classical = ols.classical

inference_rows = []
for label, result in (("Classical", ols.classical), ("HC3", ols.hc3), ("HAC, maxlags=5", ols.hac)):
    inference_rows.append({
        "covariance": label,
        "alpha estimate": float(np.asarray(result.params)[0]),
        "beta estimate": float(np.asarray(result.params)[1]),
        "beta SE": float(np.asarray(result.bse)[1]),
        "beta t": float(np.asarray(result.tvalues)[1]),
        "beta p": float(np.asarray(result.pvalues)[1]),
        "joint slope F": float(result.fvalue),
        "F p": float(result.f_pvalue),
    })
inference_table = pd.DataFrame(inference_rows).set_index("covariance")
inference_display = inference_table.copy()
for p_value_column in ("beta p", "F p"):
    inference_display[p_value_column] = inference_display[p_value_column].map(lambda value: f"{value:.3e}")
display(inference_display.round(6))
print(f"training n={int(classical.nobs)}; R2={classical.rsquared:.6f}; adjusted R2={classical.rsquared_adj:.6f}; condition={ols.design_condition_number:.3f}")
print("t/F null for the slope: beta=0. Classical inference assumes homoskedastic, serially independent errors; the constructed errors violate these assumptions.")
print("HAC maxlags=5 is a fixed teaching choice, not a bandwidth optimum.")


## 3. Multicollinearity changes coefficient allocation

A second feature is constructed as market return plus tiny noise. It is a near-duplicate teaching feature, not another observed market factor. The condition number measures numerical sensitivity, and individual slopes can move sharply while fitted values remain similar. Coefficients are conditional associations for a chosen specification; their movement does not establish causality.

In [ ]:
near_duplicate_fit = fit_ols_with_robust_covariance(
    training[["market_excess", "market_near_duplicate"]],
    training["asset_excess"],
    hac_maxlags=HAC_MAXLAGS,
)
print(f"near-duplicate design condition number={near_duplicate_fit.design_condition_number:.3f}")
print("two-feature OLS slopes:", near_duplicate_fit.classical.params.iloc[1:].to_numpy())
print("one-feature OLS slope:", classical.params["market_excess"])
print("Specification-dependent coefficient differences are not evidence of causal effects.")


## 4. Select regularization using only past training data

The three searches receive only rows 1–280. `TimeSeriesSplit` preserves order and leaves five observations between each training fold and its validation fold. Scaling is inside the pipeline, so each fold estimates its scaling values from that fold's training observations. The best estimator is refit on the complete 280-row training window after the CV choice.

In [ ]:
train_features = training[["market_excess", "market_near_duplicate"]]
selected = select_regularized_models(
    train_features,
    training_target,
    n_splits=5,
    gap=CV_GAP,
    test_size=30,
)
cv_rows = []
for name, result in selected.items():
    cv_rows.append({"model": name, "best params": result.best_params, "mean CV MSE": result.best_cv_mse})
    assert result.training_n == TRAIN_N
    assert result.training_end == training.index[-1]
    assert all(fold.gap_n == CV_GAP for fold in result.folds)
display(pd.DataFrame(cv_rows).set_index("model").round(10))
print("fold boundaries (same across models):")
for number, fold in enumerate(next(iter(selected.values())).folds, start=1):
    print(f"fold {number}: train n={fold.train_n} through {fold.train_end.date()}, gap={fold.gap_n}, validate n={fold.validation_n} from {fold.validation_start.date()}")


## 5. One final holdout evaluation

The reserved last 120 observations are used only now to compute MSE. This one comparison does not turn the simulated dataset into empirical evidence and does not select a winner.

In [ ]:
holdout_features = holdout[["market_excess", "market_near_duplicate"]]
ols_holdout_features = sm.add_constant(holdout[["market_excess"]], has_constant="add")
predictions = {
    "Mean baseline": np.full(len(holdout), float(training_target.mean())),
    "OLS (one factor)": ols.classical.predict(ols_holdout_features).to_numpy(),
}
for name, result in selected.items():
    predictions[name] = result.estimator.predict(holdout_features)
holdout_mse = {
    name: float(mean_squared_error(holdout["asset_excess"], predicted))
    for name, predicted in predictions.items()
}
mse_table = pd.DataFrame({"n": len(holdout), "holdout MSE (return/day)^2": holdout_mse})
display(mse_table)
print("No hyperparameter or model was chosen from the holdout results.")
print("MSE is forecast error on constructed observations, not net strategy performance.")


## 6. Fitted values, residual diagnostic, and held-out predictions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8.5), constrained_layout=True)
fitted = ols.classical.fittedvalues
residuals = ols.classical.resid
axes[0, 0].plot(training.index, training["asset_excess"], color="#235789", linewidth=1.0, label="Observed")
axes[0, 0].plot(training.index, fitted, color="#C1292E", linewidth=1.0, label="OLS fitted")
axes[0, 0].set_title(f"Training observations and OLS fit (n={len(training)})")
axes[0, 0].set_ylabel(Y_UNIT)
axes[0, 0].set_xlabel("UTC observation date")
axes[0, 0].legend(frameon=False)
axes[0, 0].grid(alpha=0.2)

axes[0, 1].scatter(fitted, residuals, color="#235789", s=18, alpha=0.65)
axes[0, 1].axhline(0.0, color="#C1292E", linestyle="--", linewidth=1.0)
axes[0, 1].set_title("Training residual diagnostic")
axes[0, 1].set_xlabel("OLS fitted excess return (decimal per day)")
axes[0, 1].set_ylabel("OLS residual (decimal per day)")
axes[0, 1].grid(alpha=0.2)

colors = {"Mean baseline":"#6C757D", "OLS (one factor)":"#C1292E", "Ridge":"#2A9D8F", "Lasso":"#E9C46A", "ElasticNet":"#8A5FBF"}
axes[1, 0].plot(holdout.index, holdout["asset_excess"], color="#1D3557", linewidth=1.3, label="Observed")
for name, values in predictions.items():
    axes[1, 0].plot(holdout.index, values, color=colors[name], linewidth=0.95, label=name)
axes[1, 0].set_title(f"Final chronological holdout predictions (n={len(holdout)})")
axes[1, 0].set_ylabel(Y_UNIT)
axes[1, 0].set_xlabel("UTC observation date")
axes[1, 0].legend(frameon=False, ncol=2, fontsize=8)
axes[1, 0].grid(alpha=0.2)

labels = list(holdout_mse)
compact_labels = {"Mean baseline":"Mean", "OLS (one factor)":"OLS-1F", "Ridge":"Ridge", "Lasso":"Lasso", "ElasticNet":"ElasticNet"}
axes[1, 1].bar([compact_labels[label] for label in labels], [holdout_mse[label] for label in labels], color=[colors[label] for label in labels])
axes[1, 1].set_title("Holdout MSE; lower is smaller")
axes[1, 1].set_ylabel("MSE ((decimal return per day)²)")
axes[1, 1].grid(axis="y", alpha=0.2)
fig.suptitle(f"Chapter 8 — constructed observations only; seed {SEED}; train n={len(training)}, holdout n={len(holdout)}")
for time_axis in (axes[0, 0], axes[1, 0]):
    locator = AutoDateLocator(minticks=4, maxticks=6)
    time_axis.xaxis.set_major_locator(locator)
    time_axis.xaxis.set_major_formatter(ConciseDateFormatter(locator))
figure_path = RUN_DIR / "figures" / "a2" / "ch08_regression.png"
figure_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(figure_path, dpi=170, bbox_inches="tight")
plt.close(fig)
display(Image(filename=str(figure_path)))
print(f"Saved figure: {figure_path.relative_to(ROOT)}")


## Checks and limitations

- The known no-noise test recovers the specified intercept and slope exactly within floating-point tolerance.
- Classical t/F inference relies on the linear-model assumptions; heteroskedasticity and serial dependence can invalidate conventional standard errors. HC3 addresses heteroskedasticity; HAC can address both forms of dependence asymptotically, but its lag choice and assumptions still matter.
- Ridge/Lasso/ElasticNet penalties can stabilize or shrink collinear coefficients. Standardized coefficients are not on the original raw-feature scale. A lower holdout MSE here is only a result on this fixed constructed path.
- `TimeSeriesSplit` is applied to equally spaced timestamps, with a fixed five-observation gap. Hyperparameters are selected only on the training window and no shuffled K-fold is used.
- A coefficient is a conditional association for a particular feature set. Neither its sign nor its statistical significance identifies a causal effect.
- There is no real security, no risk-free or market source series, no portfolio, and no transaction-cost model. Nothing here demonstrates tradability or CAPM validity.
